# EDA for Marathos' data

In [0]:
# --- Data from bronze table --- 
df = spark.sql("SELECT * FROM marathos_cat.bronze.raw_marathos;")

display(df)

In [0]:
#df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(BASE_DIR) do i need to do this?
df.show(3)

In [0]:
display(df.limit(100))

In [0]:
print(f"They are {df.count()} rows and {len(df.columns)} columns")
print(df.dtypes)

In [0]:
df.printSchema()


In [0]:
# --- Using .summary() and .display() to see the results with many nulls (not numeric values) ---
df.summary().display()

In [0]:
# ---    ---
df.select(
    [column
    for column, type_ in df.dtypes
    if type_ in ("int", "bigint", "double", "decimal")
]
).summary().display()

From the above, we have learnt:
- dates are from 1798 to 2022
- "Athlete year of birth" is interesting and needs to be cleaned at the silver stage

## Checking some columns 

In [0]:
# --- Event distance/length ---
df.select("Event distance/length").distinct().limit(25).display()

# Will suppress rows containing "Etappen" in the silver phase
# will change "length" for "duration" in the silver layer

In [0]:
df.select("Athlete performance").distinct().limit(25).display()


In [0]:
df.select("Athlete average speed").distinct().limit(3).display()

# Rounded at the silver stage

## The nulls

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

# --- In each column, when value is null, True is converted into 1 and the 1 are added ---
null_counts = df.select(
    [spark_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns])

null_counts = null_counts.collect()[0].asDict()                # .collect() does the math + convert into dict
[(c, nulls) for c, nulls in null_counts.items() if nulls > 0]  # looping in the dict and only count if null

In [0]:
df.select("Event name").distinct().count()


In [0]:
# --- Women and men together representation --- 
all_runners = df.select("Athlete age category").distinct().orderBy("Athlete age category", ascending=False)
display(all_runners)


In [0]:
# --- Graph ---
fig = (df.groupBy("Athlete age category")
   .count()
   .orderBy("count")
   .plot(kind="bar", x="Athlete age category", y="count", title="Runners by Age Category"))

# Source for "fig.update_traces(texttemplate='%{y} is https://www.geeksforgeeks.org/data-visualization/# add-data-labels-to-plotly-line-graph-in-python/
fig.update_traces(texttemplate='%{y}', textposition='outside')

fig.show()

In [0]:
# --- Women only ---
female_runners = df.filter(col("Athlete gender") == "F").select("Athlete age category").distinct().orderBy("Athlete age category", ascending=False)
display(female_runners)

# Quality issue, result below shows a "F35", will be interpreted as standing for "Female35" and 
# converted into "W35" at the silver stage

In [0]:
# --- Women graph ---
fig_w = (df.filter(col("Athlete gender") == "F")
         .groupBy("Athlete age category")
         .count()
         .orderBy("count")
         .plot(kind="bar", x="Athlete age category", y="count", title="Womens Age Categories"))

fig_w.update_traces(texttemplate='%{y}', textposition='outside')

display(fig_w)


In [0]:
# --- Men only ---
male_runners = df.filter(col("Athlete gender") == "M").select("Athlete age category").distinct().orderBy("Athlete age category", ascending=False)
display(male_runners)

# Quality issue, result below shows "W50" and "W45". Wil count how many they are for each and if only 1 for each, will exclude them later in the silver layer

In [0]:
# --- Men graph ---
fig_m = (df.filter(col("Athlete gender") == "M")
         .groupBy("Athlete age category")
         .count()
         .orderBy("count")
         .plot(kind="bar", x="Athlete age category", y="count", title="Men Age Categories"))

fig_m.update_traces(texttemplate='%{y}', textposition='outside')

display(fig_m)


In [0]:
# --- Comparing men and women in barchart ---
comparison_df = (df.groupBy("Athlete age category", "Athlete gender")
.count()
.orderBy("Athlete age category"))

fig_compare = comparison_df.plot(
kind="bar", 
x="Athlete age category", 
y="count", 
color="Athlete gender",  
title="Comparison by Age and Gender")

# Because of the letters "W" and "M" in the age categories, the bars for men and women for a same category cannot be side by side. 
# Will be cleaned in the silver layer.
# "F" was mentioned earlier

display(fig_compare)

In [0]:
#df.limit(1).display()

# --- Had .display() at first but it returned "None" when printing --- 
count_countries = df.groupBy("Athlete country").count().orderBy("count", ascending=False)

# --- To show the table result ---
count_countries.display()

# --- count of the countries ---
total_countries = count_countries.count()
print(f"They are {total_countries} represented countries")